# 📓 Session 6 · Exercises
### Penalised regression

The lecture loaded a table with nineteen columns and showed what happens when ordinary least squares is given all of them: a better fit on the training rows and a worse forecast. These exercises start one step earlier, by building that table yourself, and then work through the cure: a penalty on the size of the coefficients, the scaling that penalty needs, a pipeline to hold the two together, and a grid search to choose how strong the penalty should be.

By the end you will have run the whole workflow on every instrument in the data, and found out on which of them nineteen penalised columns beat one unpenalised one.

## How to use this notebook

- Run the **setup cell** below first. It loads the price data and the lecture's table, and imports the scikit-learn pieces.
- Each exercise has a **task**, then a **code cell** for your work. Cells with `...` are blanks to fill in. Replace them with real code.
- Stuck? Open the **💡 Hint**, but only after a genuine attempt. Open the **✅ Solution** to *check* yourself, not to skip the thinking.
- Every cell runs cleanly even with the blanks still in place, so pressing **Run all** never floods you with errors.
- Most exercises stand alone. A few short runs build on each other (A2 to A6, B1 to B4, C3 to C4, D1 to D2, D3 to D6, E1 to E4, F3 to F5, G1 to G3); the task says which earlier exercise it continues from. If one defeats you, open its solution, run it, and carry on. Section K uses the function from A7.

**You are not expected to finish all of these.** Do what you can, and come back to the rest when you revise. Short on time? Read the hint, then the solution. A worked solution you genuinely understand is real learning too.

**Returns are in percent here**, exactly as in the lecture, so an error of `0.22` means 0.22 percentage points.

### Difficulty

| badge | what to expect |
|:--|:--|
| ★☆☆☆☆ | One step, straight from the lecture. You are checking that you can type it. |
| ★★☆☆☆ | The same idea on new data, or two steps in a row. Nothing to decide. |
| ★★★☆☆ | Combine two ideas, or adapt a pattern rather than copy it. |
| ★★★★☆ | You choose the approach. Several steps, and something has to be worked out before you type. |
| ★★★★★ | A genuine puzzle: an insight, or a constraint that rules out the obvious route. Always solvable with what you have. |

The stars rate the work against **this** session. A three-star task here assumes everything from Sessions 1 to 5, so it is a bigger piece of work than a three-star task in an earlier notebook.

Some exercises also carry a **revisits** tag. Those need something from an earlier session as well as today's material, and they are there on purpose: the skills are meant to accumulate.

## 🧰 Your toolkit for today

Everything from Sessions 1 to 5 still applies. This card holds what Session 6 added.

> Names in brackets (`frame`, `columns`, `pipe`, ...) are **placeholders**: put your own variable there. **Hover any tool** to see what it does.

<p style="line-height:2.1"><strong>Penalised models</strong><br>
<code style="cursor:help" title="Linear regression with a charge of alpha times the sum of squared slopes. alpha=0 is ordinary least squares.">Ridge(alpha=1000)</code> &nbsp;&nbsp; <code style="cursor:help" title="The same, with a charge on the sum of absolute slopes. Sets some coefficients to exactly zero.">Lasso(alpha=0.1)</code> &nbsp;&nbsp; <code style="cursor:help" title="Both penalties at once. l1_ratio=1 is the lasso, l1_ratio=0 is ridge.">ElasticNet(alpha=0.1, l1_ratio=0.5)</code> &nbsp;&nbsp; <code style="cursor:help" title="Fit, predict, coef_ and intercept_ work exactly as for LinearRegression.">.fit(X, y)  ·  .predict(X)  ·  .coef_</code></p>

<p style="line-height:2.1"><strong>Putting columns on the same scale</strong><br>
<code style="cursor:help" title="Learns each column's mean and standard deviation. Fit it on the TRAINING rows only.">StandardScaler().fit(X_train)</code> &nbsp;&nbsp; <code style="cursor:help" title="Subtract the learned mean and divide by the learned standard deviation. Returns a NumPy array.">scaler.transform(X)</code> &nbsp;&nbsp; <code style="cursor:help" title="The numbers the scaler learned, one per column, in column order.">scaler.mean_  ·  scaler.scale_</code></p>

<p style="line-height:2.1"><strong>Two steps in one object</strong><br>
<code style="cursor:help" title="A list of (name, step) pairs. The last step is the model; the ones before it transform.">Pipeline([('scale', StandardScaler()), ('ridge', Ridge(alpha=1000))])</code> &nbsp;&nbsp; <code style="cursor:help" title="A pipeline fits, predicts and cross-validates like any model. The scaler is refitted inside every fold.">pipe.fit(X, y)  ·  pipe.predict(X)</code> &nbsp;&nbsp; <code style="cursor:help" title="The fitted object inside one step, so you can read its coef_.">pipe.named_steps['ridge']</code></p>

<p style="line-height:2.1"><strong>Choosing a setting</strong><br>
<code style="cursor:help" title="The setting to search and the values to try. step name, two underscores, argument name.">{'ridge__alpha': [1, 10, 100, 1000, 10000]}</code> &nbsp;&nbsp; <code style="cursor:help" title="Fit every value on every fold, keep the best mean score, refit it on all the training rows.">GridSearchCV(pipe, grid, cv=folds, scoring='neg_root_mean_squared_error')</code> &nbsp;&nbsp; <code style="cursor:help" title="The winning setting, its mean score (negative, for the usual reason), and the refitted winner.">search.best_params_  ·  search.best_score_  ·  search.best_estimator_</code> &nbsp;&nbsp; <code style="cursor:help" title="The whole grid as a dictionary of lists. Wrap it in pd.DataFrame to read it.">search.cv_results_</code> &nbsp;&nbsp; <code style="cursor:help" title="Every argument of an object and its current value.">model.get_params()</code></p>

<p style="line-height:2.1"><strong>Two helpers</strong><br>
<code style="cursor:help" title="Values spaced by a constant factor: here 1, 10, 100, 1000, 10000. The right shape for a grid of alphas.">np.logspace(0, 4, 5)</code> &nbsp;&nbsp; <code style="cursor:help" title="A logarithmic axis, so factors of ten are evenly spaced.">ax.set_xscale('log')</code></p>

**Formulas you will reach for**

| what | formula |
|:--|:--|
| Ordinary least squares | $$\text{RSS}=\sum_i (y_i-\hat{y}_i)^2$$ |
| Ridge | $$\text{RSS}+\alpha\sum_j \beta_j^2$$ |
| Lasso | $$\text{RSS}+\alpha\sum_j |\beta_j|$$ |
| Elastic net | $$\text{RSS}+\alpha\left[\rho\sum_j |\beta_j| + \tfrac{1-\rho}{2}\sum_j \beta_j^2\right]$$ |
| Standardising | $$z=\dfrac{x-\bar{x}}{s}$$ |
| Root mean squared error | $$\text{RMSE}=\sqrt{\dfrac{1}{n}\sum_i (y_i-\hat{y}_i)^2}$$ |


---

## ⚙️ Setup: run this first

This loads the price data and the lecture's nineteen-column table, splits it by date, and imports the scikit-learn pieces. If you are in Google Colab it downloads the data by itself.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score, TimeSeriesSplit, GridSearchCV

CANDIDATE_DIRS = ["data", os.path.join("..", "data"), "."]
REPO_RAW_URL = "https://raw.githubusercontent.com/theill95/mlfin-2026/main/data/"   # used when the CSV files are not next to the notebook


def load_csv(filename, **kwargs):
    """Read one of the course CSV files, wherever it happens to be."""
    for folder in CANDIDATE_DIRS:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return pd.read_csv(path, **kwargs)
    if REPO_RAW_URL is not None:
        return pd.read_csv(REPO_RAW_URL + filename, **kwargs)
    raise FileNotFoundError(
        f"Could not find {filename}. Run this notebook from the course folder, "
        f"upload the CSV into Colab, or set REPO_RAW_URL."
    )


def rmse(actual, predicted):
    """Root mean squared error, as in the lecture, as a plain number."""
    return float(np.sqrt(mean_squared_error(actual, predicted)))


# Eleven instruments, 2015 to 2024. Returns in PERCENT, as in the lecture.
prices = load_csv("prices.csv", parse_dates=["date"])
wide = prices.pivot(index="date", columns="ticker", values="close")
rets = wide.pct_change().dropna() * 100

# The lecture's table: the index's own history plus every stock's volatility
table = load_csv("market_features.csv", parse_dates=["date"]).set_index("date")
train = table.loc[:"2022-12-31"]
test = table.loc["2023-01-01":]
columns = list(table.columns[:-1])

folds = TimeSeriesSplit(n_splits=5)

print("table:", table.shape, "rows x columns")
print("train:", len(train), " test:", len(test), " columns:", len(columns))
print(columns)

---

## 🧱 A · Build the table yourself

The lecture loaded `market_features.csv`. Here you build it from the price file, one group of columns at a time, each exercise adding to the last, and check that the two agree.

### A1 · The index's returns  ★☆☆☆☆

Take the daily returns of the index, `SPY`, out of `rets` into a Series called `spy`.

In [ ]:
spy = ...
spy

<details>
<summary>💡 Hint</summary>

`rets` has one column per ticker. Select a column by name.

</details>

<details>
<summary>✅ Solution</summary>

```python
spy = rets['SPY']
spy
```

One return per trading day, in percent, 2,515 of them.

</details>

---

### A2 · Six volatility windows in a loop  ★★☆☆☆  · revisits S2

Build an empty DataFrame called `own`, then loop over the windows `[5, 10, 20, 40, 60, 120]` and add a column `vol_<w>d` for each: the standard deviation of the last `w` returns of `SPY`.

In [ ]:
own = pd.DataFrame()

for w in [5, 10, 20, 40, 60, 120]:
    ...

print(list(own.columns))

<details>
<summary>💡 Hint 1</summary>

The column name is text built from the number: `'vol_' + str(w) + 'd'`.

</details>

<details>
<summary>💡 Hint 2</summary>

Inside the loop: `own['vol_' + str(w) + 'd'] = rets['SPY'].rolling(w).std()`. Assigning to a name that does not exist yet creates the column.

</details>

<details>
<summary>✅ Solution</summary>

```python
own = pd.DataFrame()

for w in [5, 10, 20, 40, 60, 120]:
    own['vol_' + str(w) + 'd'] = rets['SPY'].rolling(w).std()

print(list(own.columns))
```

Six columns from three lines. `str(w)` turns the number into text so it can be glued into the name, which is the string arithmetic from Session 1 doing real work.

</details>

---

### A3 · Three return windows  ★★☆☆☆  · revisits S2

Add three more columns to your `own` from A2, in another loop: `ret_<w>d` for `w` in `[5, 20, 60]`, the **average** return over the last `w` days.

In [ ]:
for w in [5, 20, 60]:
    ...

print(list(own.columns))

<details>
<summary>💡 Hint</summary>

The same shape as A2 with `.mean()` in place of `.std()` and `'ret_'` in place of `'vol_'`.

</details>

<details>
<summary>✅ Solution</summary>

```python
for w in [5, 20, 60]:
    own['ret_' + str(w) + 'd'] = rets['SPY'].rolling(w).mean()

print(list(own.columns))
```

Nine columns. A falling market and a volatile one tend to go together, which is why the return windows earn their place next to the volatility ones.

</details>

---

### A4 · Every other stock's volatility  ★★★☆☆  · revisits S2

Add one column per stock to `own`: `<ticker>_vol`, the 20-day volatility of that stock's returns, for every ticker in `rets` **except** `SPY`.

In [ ]:
for t in rets.columns:
    ...

print(len(own.columns), 'columns')

<details>
<summary>💡 Hint 1</summary>

Loop over `rets.columns` and skip one of them with `if t != 'SPY':`.

</details>

<details>
<summary>💡 Hint 2</summary>

The column is `own[t + '_vol'] = rets[t].rolling(20).std()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
for t in rets.columns:
    if t != 'SPY':
        own[t + '_vol'] = rets[t].rolling(20).std()

print(len(own.columns), 'columns')
```

Nineteen columns. The `if` inside the loop is the only new thing here, and it is the Session 2 way of saying "all but one".

</details>

---

### A5 · The target, and the finished table  ★★☆☆☆

Add the target `vol_next` to `own`, the volatility of the **next** twenty days, then drop every incomplete row. Print the shape.

$$\text{vol\_next}_t = \text{sd}\big(r_{t+1}, \ldots, r_{t+20}\big)$$

In [ ]:
own['vol_next'] = ...
own = ...
print(...)

<details>
<summary>💡 Hint</summary>

The target is the 20-day rolling standard deviation shifted **up** by twenty rows: `.shift(-20)`. Then `.dropna()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
own['vol_next'] = rets['SPY'].rolling(20).std().shift(-20)
own = own.dropna()
print(own.shape)
```

2,376 rows and 20 columns, the same as the lecture's table. The 120-day window is what costs the most rows at the start; the target costs twenty at the end.

</details>

---

### A6 · Prove it is the same table  ★★★☆☆  · revisits S3

The setup cell loaded the lecture's `table`. Check that your `own` matches it: the largest absolute difference across every cell should be zero, or as near as floating point gets.

In [ ]:
largest_gap = ...
print(largest_gap)

<details>
<summary>💡 Hint 1</summary>

Subtracting two tables with the same index and columns subtracts cell by cell. `(own - table).abs()` is every gap.

</details>

<details>
<summary>💡 Hint 2</summary>

`.max()` on a table gives the largest per column; `.max()` again gives the largest overall. Select `table[own.columns]` first so the columns line up.

</details>

<details>
<summary>✅ Solution</summary>

```python
largest_gap = (own - table[own.columns]).abs().max().max()
print(largest_gap)
```

About 9e-16, which is floating-point noise from writing the file to text and reading it back. No loop over cells was needed: the subtraction, the absolute value and both maxima are vectorised.

</details>

---

### A7 · A function that builds it for any ticker  ★★★★☆  · revisits S2

Wrap A2 to A5 into `build_table(ticker)`: the six volatility windows and the three return windows of that ticker, the 20-day volatility of every **other** instrument, the target, and `dropna()`. Return the table.

Check it on `'SPY'` and on `'AAPL'`.

In [ ]:
def build_table(ticker):
    ...

spy_table = build_table('SPY')
apple_table = build_table('AAPL')
print(spy_table)

<details>
<summary>💡 Hint 1</summary>

Replace `'SPY'` by `ticker` everywhere, including in the `if`.

</details>

<details>
<summary>💡 Hint 2</summary>

The last line of the function is `return frame.dropna()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
def build_table(ticker):
    frame = pd.DataFrame()
    for w in [5, 10, 20, 40, 60, 120]:
        frame['vol_' + str(w) + 'd'] = rets[ticker].rolling(w).std()
    for w in [5, 20, 60]:
        frame['ret_' + str(w) + 'd'] = rets[ticker].rolling(w).mean()
    for t in rets.columns:
        if t != ticker:
            frame[t + '_vol'] = rets[t].rolling(20).std()
    frame['vol_next'] = rets[ticker].rolling(20).std().shift(-20)
    return frame.dropna()

print(build_table('SPY').shape)
print(build_table('AAPL').shape)
```

Both are 2,376 by 20. The function is the shape the last section of this notebook needs, where the whole workflow runs once per instrument.

</details>

---

## 🔍 B · Nineteen columns, and what OLS does with them

The overfitting from last time, at a larger scale, and a look at why. B1 to B4 share the two models B1 fits.

### B1 · One column, all columns  ★★☆☆☆

Fit two linear regressions on the training rows: one on `vol_20d` alone, one on all nineteen `columns`. Print the **test** RMSE of each.

In [ ]:
one = LinearRegression()
...

ols = LinearRegression()
...

print('one column :', ...)
print('all columns:', ...)

<details>
<summary>💡 Hint</summary>

The setup cell defines `rmse(actual, predicted)`. Remember the double brackets for a single column: `train[['vol_20d']]`.

</details>

<details>
<summary>✅ Solution</summary>

```python
one = LinearRegression()
one.fit(train[['vol_20d']], train['vol_next'])

ols = LinearRegression()
ols.fit(train[columns], train['vol_next'])

print('one column :', rmse(test['vol_next'], one.predict(test[['vol_20d']])))
print('all columns:', rmse(test['vol_next'], ols.predict(test[columns])))
```

0.2369 against 0.2481. Eighteen extra columns, and the forecast got worse.

</details>

---

### B2 · And on the training rows  ★★☆☆☆

Now print the **training** RMSE of `one` and `ols` from B1. Which is lower, and what does that say next to B1?

In [ ]:
print('one column :', ...)
print('all columns:', ...)

<details>
<summary>💡 Hint</summary>

Predict on `train[...]` instead of `test[...]`, and score against `train['vol_next']`.

</details>

<details>
<summary>✅ Solution</summary>

```python
print('one column :', rmse(train['vol_next'], one.predict(train[['vol_20d']])))
print('all columns:', rmse(train['vol_next'], ols.predict(train[columns])))
```

0.5877 against 0.5199: the nineteen-column model fits the training rows better and forecasts worse. That gap between training and test error is what overfitting looks like in numbers.

</details>

---

### B3 · Draw the nineteen coefficients  ★★★☆☆  · revisits S3

Draw a horizontal bar chart of the coefficients of `ols` from B1, one bar per column, with the column names on the axis.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
...
plt.show()

<details>
<summary>💡 Hint 1</summary>

`ax.barh(columns, ols.coef_)` draws one bar per name.

</details>

<details>
<summary>💡 Hint 2</summary>

`ax.axvline(0, color='black', linewidth=1)` marks zero, and `ax.invert_yaxis()` puts the first column at the top.

</details>

<details>
<summary>✅ Solution</summary>

```python
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(columns, ols.coef_)
ax.axvline(0, color='black', linewidth=1)
ax.invert_yaxis()
ax.set_xlabel('coefficient')
ax.set_title('Nineteen columns, ordinary least squares', loc='left')
plt.show()
```

Bars pointing both ways, with the volatility of some stocks lowering the forecast of market volatility. The chart is the quickest way to see that something is wrong with the coefficients even before you read them.

</details>

---

### B4 · Count the wrong signs  ★★★☆☆  · revisits S3

Every **volatility** column in this table should, if anything, raise the forecast when it rises. Count how many coefficients of `ols` are negative, and print their names. Which of them are volatility columns?

In [ ]:
n_negative = ...
print(n_negative)

...

<details>
<summary>💡 Hint 1</summary>

`ols.coef_ < 0` is an array of True and False; `.sum()` counts the Trues.

</details>

<details>
<summary>💡 Hint 2</summary>

`for name, b in zip(columns, ols.coef_):` with `if b < 0: print(name)` inside.

</details>

<details>
<summary>✅ Solution</summary>

```python
n_negative = (ols.coef_ < 0).sum()
print(n_negative)

for name, b in zip(columns, ols.coef_):
    if b < 0:
        print(name)
```

7 of the nineteen are negative. The two return columns are allowed to be, since a falling market is a volatile one. The other 5, vol_40d, vol_120d, JPM_vol, KO_vol, JNJ_vol, are volatility columns with the wrong sign. The mask counts in one line; the loop names.

</details>

---

### B5 · Against the two rules that need no model  ★★★☆☆  · revisits S4

Score the two baselines from last time on the test rows: guessing the training average, and repeating the last twenty days (`vol_20d`). Where do the numbers from B1 sit among the four?

In [ ]:
guess = ...
print('guess the average :', ...)
print('repeat this month :', ...)

<details>
<summary>💡 Hint 1</summary>

The average is `train['vol_next'].mean()`, repeated with `np.full(len(test), guess)`.

</details>

<details>
<summary>💡 Hint 2</summary>

The persistence forecast is simply the column `test['vol_20d']`.

</details>

<details>
<summary>✅ Solution</summary>

```python
guess = train['vol_next'].mean()
print('guess the average :', rmse(test['vol_next'], np.full(len(test), guess)))
print('repeat this month :', rmse(test['vol_next'], test['vol_20d']))
```

0.2906 and 0.2604. The nineteen-column OLS, at 0.2481, still beats both rules, so it is not useless. It is worse than the single column, which is the point.

</details>

---

### B6 · How alike are the columns  ★★★☆☆  · revisits S4

Compute the correlation between `vol_40d` and `vol_60d` on the training rows. Then find the **most** correlated pair among all nineteen columns.

In [ ]:
corr_40_60 = ...
print(corr_40_60)

corr_table = ...
corr_table

<details>
<summary>💡 Hint 1</summary>

`train['vol_40d'].corr(train['vol_60d'])` for one pair; `train[columns].corr()` for all of them at once.

</details>

<details>
<summary>💡 Hint 2</summary>

Reading the biggest off-diagonal entry by eye is fine. `.round(2)` makes the table easier to scan.

</details>

<details>
<summary>✅ Solution</summary>

```python
corr_40_60 = train['vol_40d'].corr(train['vol_60d'])
print(corr_40_60)

corr_table = train[columns].corr().round(2)
corr_table
```

0.93 for the pair in the question, and of the 171 pairs in the table the most correlated is `vol_40d` with `vol_60d` at 0.93. Two columns that alike can trade coefficient between them almost freely, which is where the wrong signs come from.

</details>

---

### B7 · The same coefficient on five different blocks  ★★★★☆  · revisits S5

Fit the nineteen-column OLS on the **fitting rows of each fold** of `folds` and collect the coefficient on `vol_120d` in a list. Print the list.

In [ ]:
spot = columns.index('vol_120d')
coefs_120 = []

for fit_rows, score_rows in folds.split(train):
    ...

print(coefs_120)

<details>
<summary>💡 Hint 1</summary>

`train.iloc[fit_rows]` is the block of rows to fit on. Fit a fresh `LinearRegression` on it each time.

</details>

<details>
<summary>💡 Hint 2</summary>

Append `round(float(model.coef_[spot]), 3)` to the list. `float()` turns the NumPy number into a plain one, which prints more neatly.

</details>

<details>
<summary>✅ Solution</summary>

```python
spot = columns.index('vol_120d')
coefs_120 = []

for fit_rows, score_rows in folds.split(train):
    block = train.iloc[fit_rows]
    model = LinearRegression()
    model.fit(block[columns], block['vol_next'])
    coefs_120.append(round(float(model.coef_[spot]), 3))

print(coefs_120)
```

From -0.94 to 1.02, changing sign along the way. A coefficient that depends this much on which years it was fitted on has no business forecasting the next year.

</details>

---

## 🎯 C · Ridge

One new argument, and what it does to the coefficients.

### C1 · Ridge in three lines  ★☆☆☆☆

Fit `Ridge(alpha=1000)` on all nineteen columns of the training rows and print its test RMSE.

In [ ]:
ridge = ...
...
print(...)

<details>
<summary>💡 Hint</summary>

`Ridge` fits and predicts exactly like `LinearRegression`.

</details>

<details>
<summary>✅ Solution</summary>

```python
ridge = Ridge(alpha=1000)
ridge.fit(train[columns], train['vol_next'])
print(rmse(test['vol_next'], ridge.predict(test[columns])))
```

0.2284, against 0.2481 for the same columns unpenalised. The only change is the first line.

</details>

---

### C2 · Alpha zero is OLS  ★★☆☆☆  · revisits S3

Fit ordinary least squares and `Ridge(alpha=0)` on all nineteen columns, and print the two test RMSEs. Then print whether they agree to six decimals.

In [ ]:
plain = ...
...
plain_rmse = ...

ridge0 = ...
...
ridge0_rmse = ...

print(plain_rmse, ridge0_rmse)
print(...)

<details>
<summary>💡 Hint</summary>

`abs(a - b) < 0.000001` is True or False.

</details>

<details>
<summary>✅ Solution</summary>

```python
plain = LinearRegression()
plain.fit(train[columns], train['vol_next'])
plain_rmse = rmse(test['vol_next'], plain.predict(test[columns]))

ridge0 = Ridge(alpha=0)
ridge0.fit(train[columns], train['vol_next'])
ridge0_rmse = rmse(test['vol_next'], ridge0.predict(test[columns]))

print(plain_rmse, ridge0_rmse)
print(abs(plain_rmse - ridge0_rmse) < 0.000001)
```

True. With no penalty there is nothing to shrink, and ridge is ordinary least squares. Every number the penalty changes is therefore the penalty's doing.

</details>

---

### C3 · Alpha on a validation block  ★★★☆☆  · revisits S5

Choosing alpha needs held-out rows. Cut the training block at the end of 2020 into `fit` and `val`, then loop over `[1, 10, 100, 1000, 10000, 100000]`: fit a ridge on `fit`, score it on `val`, and print each alpha with its RMSE.

In [ ]:
fit = train.loc[:'2020-12-31']
val = train.loc['2021-01-01':]

for alpha in [1, 10, 100, 1000, 10000, 100000]:
    ...

<details>
<summary>💡 Hint</summary>

Inside the loop: create `Ridge(alpha=alpha)`, fit on `fit[columns]`, score on `val`.

</details>

<details>
<summary>✅ Solution</summary>

```python
fit = train.loc[:'2020-12-31']
val = train.loc['2021-01-01':]

for alpha in [1, 10, 100, 1000, 10000, 100000]:
    model = Ridge(alpha=alpha)
    model.fit(fit[columns], fit['vol_next'])
    print(alpha, round(rmse(val['vol_next'], model.predict(val[columns])), 4))
```

The error falls to alpha 1000 and rises after it. The test rows were not touched, so this is a legitimate way to choose; the folds in section F are the same idea with five cuts instead of one.

</details>

---

### C4 · A function for the loop body  ★★★☆☆  · revisits S2

Write `val_rmse(alpha)`: it fits a ridge with that alpha on `fit` and returns the RMSE on `val`, both from C3. Use it to store the six results in a dictionary keyed by alpha, and print the best key.

In [ ]:
def val_rmse(alpha):
    ...

by_alpha = {}
for alpha in [1, 10, 100, 1000, 10000, 100000]:
    ...

print(by_alpha)
print('best:', ...)

<details>
<summary>💡 Hint 1</summary>

The function body is the loop body from C3 with `return` in place of `print`.

</details>

<details>
<summary>💡 Hint 2</summary>

`min(by_alpha, key=by_alpha.get)` is the key with the smallest value.

</details>

<details>
<summary>✅ Solution</summary>

```python
def val_rmse(alpha):
    model = Ridge(alpha=alpha)
    model.fit(fit[columns], fit['vol_next'])
    return rmse(val['vol_next'], model.predict(val[columns]))

by_alpha = {}
for alpha in [1, 10, 100, 1000, 10000, 100000]:
    by_alpha[alpha] = round(val_rmse(alpha), 4)

print(by_alpha)
print('best:', min(by_alpha, key=by_alpha.get))
```

Best at 1000. A function, a dictionary and `min` with a key: the Session 2 toolkit, and it is exactly what `GridSearchCV` does internally.

</details>

---

### C5 · The intercept is not penalised  ★★★☆☆

Fit ridge with alpha 1, 1000 and 10,000,000 and print each intercept. Then print the training mean of `vol_next`. What does the intercept become as the slopes are crushed?

In [ ]:
for alpha in [1, 1000, 10000000]:
    ...

print('training mean:', ...)

<details>
<summary>💡 Hint</summary>

Print `model.intercept_` inside the loop. The mean is `train['vol_next'].mean()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
for alpha in [1, 1000, 10000000]:
    model = Ridge(alpha=alpha)
    model.fit(train[columns], train['vol_next'])
    print(alpha, round(model.intercept_, 4))

print('training mean:', round(train['vol_next'].mean(), 4))
```

The intercept moves from 0.323 to 0.982, and the training mean is 0.983. With every slope at zero the forecast is the intercept, and the intercept is free to be the average. That is why the penalty leaves it alone.

</details>

---

### C6 · How much coefficient is left  ★★★★☆  · revisits S3

The ridge penalty charges for $\sum_j \beta_j^2$. Compute that sum for alpha 0, 1000 and 100000, without a loop over the coefficients.

In [ ]:
for alpha in [0, 1000, 100000]:
    model = Ridge(alpha=alpha)
    model.fit(train[columns], train['vol_next'])
    size = ...
    print(alpha, size)

<details>
<summary>💡 Hint</summary>

`model.coef_` is an array. Square it and sum it: `(model.coef_ ** 2).sum()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
for alpha in [0, 1000, 100000]:
    model = Ridge(alpha=alpha)
    model.fit(train[columns], train['vol_next'])
    size = (model.coef_ ** 2).sum()
    print(alpha, round(size, 5))
```

0.691, then 0.044, then 0.00029. The penalty term is the thing being squeezed, and you can watch it shrink. Squaring and summing a whole array at once is the vectorised habit from Session 3.

</details>

---

## 📏 D · The penalty and the units

Why a penalised model needs every column on the same scale, and how to put them there without leaking the test rows. D1 and D2 share the decimal copies D1 makes; D3 to D6 share the scaler D3 fits.

### D1 · OLS does not care about units  ★★☆☆☆  · revisits S3

Make copies of `train` and `test` in which `vol_20d` is divided by 100 (so it is in decimals). Fit OLS on both versions and print the coefficient on `vol_20d` from each. Then print the ratio.

In [ ]:
train_dec = train.copy()
test_dec = test.copy()
train_dec['vol_20d'] = train_dec['vol_20d'] / 100
test_dec['vol_20d'] = test_dec['vol_20d'] / 100
spot = columns.index('vol_20d')

ols_pct = ...
...
ols_dec = ...
...

print(..., ...)
print('ratio:', ...)

<details>
<summary>💡 Hint</summary>

Fit `ols_pct` on `train[columns]` and `ols_dec` on `train_dec[columns]`, then print `ols_pct.coef_[spot]` and `ols_dec.coef_[spot]`. The ratio is one coefficient divided by the other.

</details>

<details>
<summary>✅ Solution</summary>

```python
train_dec = train.copy()
test_dec = test.copy()
train_dec['vol_20d'] = train_dec['vol_20d'] / 100
test_dec['vol_20d'] = test_dec['vol_20d'] / 100
spot = columns.index('vol_20d')

ols_pct = LinearRegression()
ols_pct.fit(train[columns], train['vol_next'])
ols_dec = LinearRegression()
ols_dec.fit(train_dec[columns], train_dec['vol_next'])

print(ols_pct.coef_[spot], ols_dec.coef_[spot])
print('ratio:', ols_dec.coef_[spot] / ols_pct.coef_[spot])
```

0.0191 and 1.9130: a ratio of exactly 100. The coefficient absorbs the change of units and the forecast is identical.

</details>

---

### D2 · Ridge does care  ★★☆☆☆

Repeat D1 with `Ridge(alpha=1000)` on both versions, using `train_dec` and `spot` from D1. What happens to the coefficient on `vol_20d` when the column is in decimals?

In [ ]:
ridge_pct = ...
...
ridge_dec = ...
...

print(..., ...)

<details>
<summary>💡 Hint</summary>

Same code as D1 with `Ridge(alpha=1000)` in place of `LinearRegression()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
ridge_pct = Ridge(alpha=1000)
ridge_pct.fit(train[columns], train['vol_next'])
ridge_dec = Ridge(alpha=1000)
ridge_dec.fit(train_dec[columns], train_dec['vol_next'])

print(ridge_pct.coef_[spot], ridge_dec.coef_[spot])
```

0.0314 in percent, 0.00035 in decimals. The column would need a coefficient a hundred times larger, which costs ten thousand times more penalty, so ridge drops it. Nothing about its information changed.

</details>

---

### D3 · StandardScaler  ★★☆☆☆

Fit a `StandardScaler` on the training columns, transform both blocks into `X_train` and `X_test`, and print the mean and standard deviation the scaler learned for `vol_20d`.

In [ ]:
spot = columns.index('vol_20d')

scaler = ...
...
X_train = ...
X_test = ...

print(..., ...)

<details>
<summary>💡 Hint</summary>

`scaler.fit(train[columns])`, then `scaler.transform(...)` on each block. The learned numbers are `mean_` and `scale_`, one per column.

</details>

<details>
<summary>✅ Solution</summary>

```python
spot = columns.index('vol_20d')

scaler = StandardScaler()
scaler.fit(train[columns])
X_train = scaler.transform(train[columns])
X_test = scaler.transform(test[columns])

print(scaler.mean_[spot], scaler.scale_[spot])
```

A mean of 0.979 and a standard deviation of 0.693, in percent, learned from the training rows alone. `transform` returns a NumPy array, so the column names are gone and `spot` is how you find a column.

</details>

---

### D4 · The same thing by hand  ★★★☆☆  · revisits S3

Standardise `vol_20d` yourself with the formula from Session 4, using `np.std` for the standard deviation, and check the result against column `spot` of `X_train` from D3: print the largest absolute difference.

$$z = \frac{x - \bar{x}}{s}$$

In [ ]:
z_hand = ...
largest_gap = ...
print(largest_gap)

<details>
<summary>💡 Hint 1</summary>

`(train['vol_20d'] - train['vol_20d'].mean()) / np.std(train['vol_20d'])`.

</details>

<details>
<summary>💡 Hint 2</summary>

`np.abs(z_hand - X_train[:, spot]).max()`. `X_train[:, spot]` is one column of the array, as in Session 3's matrices.

</details>

<details>
<summary>✅ Solution</summary>

```python
z_hand = (train['vol_20d'] - train['vol_20d'].mean()) / np.std(train['vol_20d'])
largest_gap = np.abs(z_hand - X_train[:, spot]).max()
print(largest_gap)
```

Zero, or a number of the order 1e-16: the same numbers. `np.std` is used rather than pandas' `.std()` because the scaler divides by $n$ and pandas divides by $n - 1$; the difference is in the fourth decimal, and it is the kind of thing a check like this is for.

</details>

---

### D5 · The test rows, in training units  ★★★☆☆  · revisits S3

Print the mean and standard deviation of column `spot` of `X_test` from D3. They are not 0 and 1. Why not, and what do the numbers say about 2023 and 2024?

In [ ]:
print(..., ...)

<details>
<summary>💡 Hint</summary>

`X_test[:, spot].mean()` and `X_test[:, spot].std()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
print(X_test[:, spot].mean(), X_test[:, spot].std())
```

A mean of -0.27 and a standard deviation of 0.30. The test rows were transformed with the **training** mean and spread, as they must be, and 2023 and 2024 were calmer and steadier than the training years. A mean of exactly zero here would be the sign of a leak.

</details>

---

### D6 · Fix the leak  ★★★★☆  · revisits S5

The cell below fits a second scaler on the test rows. That uses the test block's own mean and spread, which is information from the future. Fit a ridge on `X_train` from D3, score it on the leaky array, then transform the test rows properly with `scaler` from D3 and score again.

In [ ]:
leaky = StandardScaler()
leaky.fit(test[columns])          # the leak
X_test_leaky = leaky.transform(test[columns])

ridge_sc = ...
...
print('leaky :', ...)

X_test = ...
print('proper:', ...)

<details>
<summary>💡 Hint</summary>

The proper version is one line: `scaler.transform(test[columns])`, with the scaler that was fitted on `train`.

</details>

<details>
<summary>✅ Solution</summary>

```python
leaky = StandardScaler()
leaky.fit(test[columns])          # the leak
X_test_leaky = leaky.transform(test[columns])

ridge_sc = Ridge(alpha=1000)
ridge_sc.fit(X_train, train['vol_next'])
print('leaky :', rmse(test['vol_next'], ridge_sc.predict(X_test_leaky)))

X_test = scaler.transform(test[columns])
print('proper:', rmse(test['vol_next'], ridge_sc.predict(X_test)))
```

0.3528 with the leak and 0.2181 without. The leaky version is worse here, not better, because it re-centres a calm test period as if it were average. Either way it is a number that could not have been computed on the day the forecast was made.

</details>

---

## 🔗 E · Pipeline

The scaler and the model as one object, so the folds refit both. E2 to E4 use the pipeline E1 builds.

### E1 · Scale and fit in one object  ★☆☆☆☆

Build a `Pipeline` with a `StandardScaler` step called `'scale'` and a `Ridge(alpha=1000)` step called `'ridge'`. Fit it and print the test RMSE.

In [ ]:
pipe = Pipeline([...])
...
print(...)

<details>
<summary>💡 Hint</summary>

`Pipeline([('scale', StandardScaler()), ('ridge', Ridge(alpha=1000))])`, then `.fit` and `.predict` on the raw columns. The pipeline does the scaling.

</details>

<details>
<summary>✅ Solution</summary>

```python
pipe = Pipeline([('scale', StandardScaler()), ('ridge', Ridge(alpha=1000))])
pipe.fit(train[columns], train['vol_next'])
print(rmse(test['vol_next'], pipe.predict(test[columns])))
```

0.2181, the same as scaling by hand in D6. The raw columns go in; the pipeline transforms them with its own fitted scaler before the ridge sees them.

</details>

---

### E2 · Reach inside  ★★☆☆☆  · revisits S5

Get the fitted `Ridge` out of `pipe` from E1, and print the name of the column with the largest coefficient in absolute value.

In [ ]:
coefs = ...
biggest = ...
print(biggest)

<details>
<summary>💡 Hint 1</summary>

`pipe.named_steps['ridge'].coef_`.

</details>

<details>
<summary>💡 Hint 2</summary>

`np.abs(coefs).argmax()` is the position of the largest absolute value; `columns[...]` turns it into a name.

</details>

<details>
<summary>✅ Solution</summary>

```python
coefs = pipe.named_steps['ridge'].coef_
biggest = columns[np.abs(coefs).argmax()]
print(biggest)
```

`ret_5d`, at -0.091 per standard deviation. The pipeline itself has no `coef_`; the step inside it does.

</details>

---

### E3 · Cross-validate the pipeline  ★★★☆☆  · revisits S5

Run `cross_val_score` on `pipe` with `folds`, print the five fold RMSEs and their mean, and print the number of the worst fold.

In [ ]:
scores = ...
print(...)
print('mean :', ...)
print('worst:', ...)

<details>
<summary>💡 Hint 1</summary>

Exactly as for `LinearRegression()` last time: `cross_val_score(pipe, train[columns], train['vol_next'], cv=folds, scoring='neg_root_mean_squared_error')`.

</details>

<details>
<summary>💡 Hint 2</summary>

`(-scores).argmax() + 1` is the fold number, counting from one.

</details>

<details>
<summary>✅ Solution</summary>

```python
scores = cross_val_score(pipe, train[columns], train['vol_next'],
                         cv=folds, scoring='neg_root_mean_squared_error')
print((-scores).round(3))
print('mean :', round(-scores.mean(), 4))
print('worst:', (-scores).argmax() + 1)
```

A mean of 0.5599, and fold 3 is the worst at 1.127, because it is scored on the block containing March 2020. Inside each fold the scaler was refitted on that fold's fitting rows.

</details>

---

### E4 · What refitting the scaler is worth here  ★★★★☆  · revisits S5

Do it the leaky way for comparison: fit one scaler on **all** the training rows and transform them once, then cross-validate a plain `Ridge(alpha=1000)` on the scaled array. Print that mean next to the pipeline's from E3. How large is the difference, and why?

In [ ]:
once = StandardScaler()
...
X_all = ...
leaky = ...

print('proper:', ...)
print('leaky :', ...)

<details>
<summary>💡 Hint 1</summary>

`once.fit(train[columns])`, `X_all = once.transform(train[columns])`, then `cross_val_score(Ridge(alpha=1000), X_all, train['vol_next'], cv=folds, ...)`.

</details>

<details>
<summary>💡 Hint 2</summary>

The difference is small because a mean and a standard deviation over hundreds of rows barely move when a block is added.

</details>

<details>
<summary>✅ Solution</summary>

```python
once = StandardScaler()
once.fit(train[columns])
X_all = once.transform(train[columns])
leaky = cross_val_score(Ridge(alpha=1000), X_all, train['vol_next'],
                        cv=folds, scoring='neg_root_mean_squared_error')

print('proper:', round(-scores.mean(), 4))
print('leaky :', round(-leaky.mean(), 4))
```

0.5599 against 0.5605. The leak is real and its effect is tiny, because the scaler's two numbers are stable. The pipeline costs nothing and removes the question, which is why it is the habit to build.

</details>

---

## 🎛️ F · Choosing alpha

A loop, a curve, and the object that does both for you.

### F1 · A loop over alphas  ★★☆☆☆  · revisits S2

For each alpha in `[1, 10, 100, 1000, 10000]`, build the pipeline, cross-validate it with `folds`, and store the mean RMSE in a dictionary `cv_by_alpha`. Print the dictionary and the best key.

In [ ]:
cv_by_alpha = {}

for alpha in [1, 10, 100, 1000, 10000]:
    ...

print(cv_by_alpha)
print('best:', ...)

<details>
<summary>💡 Hint 1</summary>

The loop body is E3 with a fresh pipeline, `Ridge(alpha=alpha)`, and the mean stored under `cv_by_alpha[alpha]`. Give the pipeline its own name so `pipe` from E1 is kept.

</details>

<details>
<summary>💡 Hint 2</summary>

`min(cv_by_alpha, key=cv_by_alpha.get)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
cv_by_alpha = {}

for alpha in [1, 10, 100, 1000, 10000]:
    candidate = Pipeline([('scale', StandardScaler()), ('ridge', Ridge(alpha=alpha))])
    scores = cross_val_score(candidate, train[columns], train['vol_next'],
                             cv=folds, scoring='neg_root_mean_squared_error')
    cv_by_alpha[alpha] = round(float(-scores.mean()), 4)

print(cv_by_alpha)
print('best:', min(cv_by_alpha, key=cv_by_alpha.get))
```

Best at 1000, with the error falling until then and rising after. Five fits per alpha, twenty-five in all, and the test rows untouched.

</details>

---

### F2 · The validation curve  ★★★☆☆  · revisits S3

Repeat F1 on the finer grid `np.logspace(0, 5, 11)` and draw the mean RMSE against alpha as a line, with a logarithmic x-axis.

In [ ]:
alphas = np.logspace(0, 5, 11)
errors = []

for alpha in alphas:
    ...

fig, ax = plt.subplots(figsize=(8, 3.2))
...
plt.show()

<details>
<summary>💡 Hint 1</summary>

`np.logspace(0, 5, 11)` is eleven values from 1 to 100,000, each a factor of about 3.2 apart. Append each mean to `errors`.

</details>

<details>
<summary>💡 Hint 2</summary>

`ax.plot(alphas, errors)` then `ax.set_xscale('log')`, so the factors of ten are evenly spaced.

</details>

<details>
<summary>✅ Solution</summary>

```python
alphas = np.logspace(0, 5, 11)
errors = []

for alpha in alphas:
    candidate = Pipeline([('scale', StandardScaler()), ('ridge', Ridge(alpha=alpha))])
    scores = cross_val_score(candidate, train[columns], train['vol_next'],
                             cv=folds, scoring='neg_root_mean_squared_error')
    errors.append(-scores.mean())

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(alphas, errors, marker='o')
ax.set_xscale('log')
ax.set_xlabel('alpha')
ax.set_ylabel('cross-validated RMSE')
ax.set_title('The validation curve', loc='left')
plt.show()
```

A U shape with its bottom near 1,000. Left of it the penalty is too weak and the model overfits; right of it the penalty removes the signal. The log axis is what makes the curve readable.

</details>

---

### F3 · GridSearchCV  ★★☆☆☆

Let `GridSearchCV` run F1. Build a pipeline `base` with `Ridge()` and no alpha, a grid `{'ridge__alpha': [1, 10, 100, 1000, 10000]}`, fit the search on the training rows as `search`, and print the best alpha and the best score.

In [ ]:
base = ...
grid = ...

search = ...
...

print(...)
print(...)

<details>
<summary>💡 Hint</summary>

`GridSearchCV(base, grid, cv=folds, scoring='neg_root_mean_squared_error')`, then `.fit`. The best score is negative; print `-search.best_score_`.

</details>

<details>
<summary>✅ Solution</summary>

```python
base = Pipeline([('scale', StandardScaler()), ('ridge', Ridge())])
grid = {'ridge__alpha': [1, 10, 100, 1000, 10000]}

search = GridSearchCV(base, grid, cv=folds, scoring='neg_root_mean_squared_error')
search.fit(train[columns], train['vol_next'])

print(search.best_params_)
print(-search.best_score_)
```

Alpha 1000 at 0.5599, exactly the numbers your loop in F1 produced. `'ridge__alpha'` is the step name, two underscores, then the argument.

</details>

---

### F4 · Read the whole grid  ★★☆☆☆

Turn `search.cv_results_` from F3 into a DataFrame and show the columns `param_ridge__alpha`, `mean_test_score`, `std_test_score` and `rank_test_score`, sorted by rank.

In [ ]:
results = ...
results

<details>
<summary>💡 Hint</summary>

`pd.DataFrame(search.cv_results_)`, then select the four columns with a list inside the brackets and `.sort_values('rank_test_score')`.

</details>

<details>
<summary>✅ Solution</summary>

```python
results = pd.DataFrame(search.cv_results_)
results = results[['param_ridge__alpha', 'mean_test_score', 'std_test_score', 'rank_test_score']]
results.sort_values('rank_test_score')
```

One row per alpha, with the mean score, its spread across the folds, and a rank. The spread column is the fold-to-fold variation from last time, and it is far larger than the gaps between the top ranks.

</details>

---

### F5 · Predict with the search  ★★★☆☆

Use `search` directly to predict the test rows and print the RMSE. Then print the alpha inside `search.best_estimator_`.

In [ ]:
print('test RMSE:', ...)
print('alpha    :', ...)

<details>
<summary>💡 Hint 1</summary>

`search.predict(test[columns])` uses the best pipeline, refitted on all the training rows.

</details>

<details>
<summary>💡 Hint 2</summary>

`search.best_estimator_.named_steps['ridge'].alpha` reads the setting back.

</details>

<details>
<summary>✅ Solution</summary>

```python
print('test RMSE:', rmse(test['vol_next'], search.predict(test[columns])))
print('alpha    :', search.best_estimator_.named_steps['ridge'].alpha)
```

0.2181 with alpha 1000. The search chose alpha on the folds and refitted on every training row; the test rows are opened once, here.

</details>

---

### F6 · A grid with the answer outside it  ★★★★☆  · revisits S1

Search a scaling-plus-`Ridge()` pipeline over the grid `[1, 2, 3, 4, 5]`. Print the best alpha and score, then write an `if` that prints a warning when the best value is the largest one in the grid.

In [ ]:
narrow = {'ridge__alpha': [1, 2, 3, 4, 5]}
narrow_search = ...
...

best = ...
print(best, ...)

if ...:
    print('the best value is at the edge of the grid: widen it')

<details>
<summary>💡 Hint 1</summary>

`best = narrow_search.best_params_['ridge__alpha']`.

</details>

<details>
<summary>💡 Hint 2</summary>

`if best == max(narrow['ridge__alpha']):`.

</details>

<details>
<summary>✅ Solution</summary>

```python
narrow = {'ridge__alpha': [1, 2, 3, 4, 5]}
narrow_search = GridSearchCV(Pipeline([('scale', StandardScaler()), ('ridge', Ridge())]),
                             narrow, cv=folds, scoring='neg_root_mean_squared_error')
narrow_search.fit(train[columns], train['vol_next'])

best = narrow_search.best_params_['ridge__alpha']
print(best, -narrow_search.best_score_)

if best == max(narrow['ridge__alpha']):
    print('the best value is at the edge of the grid: widen it')
```

Best at 5, the edge, with a score of 0.7056 against 0.5599 for the proper grid. A search can only pick from what it was given. A winner at the edge means the grid was wrong, not that the answer is 5, and a two-line check catches it.

</details>

---

### F7 · Grow alpha until the error turns  ★★★★★  · revisits S2

Without a grid: start at alpha 1, cross-validate, multiply alpha by ten, and keep going **while** the error keeps falling. Stop at the first rise, with a `break`, and print the last alpha that improved. The loop below already stops at 100,000 so it can never run forever.

In [ ]:
alpha = 1
previous = None
best = None

while alpha <= 100000:
    ...
    alpha = alpha * 10

print('last improvement at alpha', best)

<details>
<summary>💡 Hint 1</summary>

Compute the mean error for the current alpha. If `previous` is not `None` and the error is larger than `previous`, `break`.

</details>

<details>
<summary>💡 Hint 2</summary>

Otherwise store the error in `previous`, remember this alpha as the best so far, and multiply alpha by ten.

</details>

<details>
<summary>✅ Solution</summary>

```python
alpha = 1
previous = None
best = None

while alpha <= 100000:
    candidate = Pipeline([('scale', StandardScaler()), ('ridge', Ridge(alpha=alpha))])
    scores = cross_val_score(candidate, train[columns], train['vol_next'],
                             cv=folds, scoring='neg_root_mean_squared_error')
    error = -scores.mean()
    print(alpha, round(error, 4))
    if previous is not None and error > previous:
        break
    previous = error
    best = alpha
    alpha = alpha * 10

print('last improvement at alpha', best)
```

It stops after alpha 10,000, and the last improvement was at 1,000. A `while` loop with a `break` is the Session 2 tool for "keep going until something happens", and it finds the bottom of the validation curve without deciding the grid in advance.

</details>

---

## ✂️ G · Lasso

The penalty that sets coefficients to exactly zero.

### G1 · Lasso in a pipeline  ★☆☆☆☆

Build and fit a pipeline with a `StandardScaler` and `Lasso(alpha=0.1)`, and print the test RMSE.

In [ ]:
lasso = Pipeline([...])
...
print(...)

<details>
<summary>💡 Hint</summary>

Same as E1 with `Lasso(alpha=0.1)` as the second step, named `'lasso'`.

</details>

<details>
<summary>✅ Solution</summary>

```python
lasso = Pipeline([('scale', StandardScaler()), ('lasso', Lasso(alpha=0.1))])
lasso.fit(train[columns], train['vol_next'])
print(rmse(test['vol_next'], lasso.predict(test[columns])))
```

0.2166, level with ridge at 0.2181.

</details>

---

### G2 · Count the zeros  ★★☆☆☆

Get the coefficients out of `lasso` from G1 and count how many are exactly zero.

In [ ]:
coefs = ...
n_zero = ...
print(n_zero, 'of', len(columns))

<details>
<summary>💡 Hint</summary>

`lasso.named_steps['lasso'].coef_`, then `(coefs == 0).sum()`. With ridge that count would be zero.

</details>

<details>
<summary>✅ Solution</summary>

```python
coefs = lasso.named_steps['lasso'].coef_
n_zero = (coefs == 0).sum()
print(n_zero, 'of', len(columns))
```

14 of 19. Not small; exactly zero. Those columns play no part in the forecast.

</details>

---

### G3 · Which columns survived  ★★☆☆☆  · revisits S2

Print the name and coefficient of every column the lasso kept, to three decimals, using `coefs` from G2.

In [ ]:
...

<details>
<summary>💡 Hint</summary>

`for name, b in zip(columns, coefs):` with `if b != 0: print(name, round(b, 3))` inside.

</details>

<details>
<summary>✅ Solution</summary>

```python
for name, b in zip(columns, coefs):
    if b != 0:
        print(name, round(b, 3))
```

vol_5d, vol_10d, ret_5d, AAPL_vol, XOM_vol: the last few days of the index's own history, its recent return, and two of the stocks. Five columns out of nineteen, chosen by the penalty rather than by hand.

</details>

---

### G4 · Lasso's alpha lives on another scale  ★★★☆☆

Build a lasso pipeline with `Lasso()` and no alpha, and run a grid search over `[0.001, 0.01, 0.1, 1]` as `lasso_search`. Print the best alpha and score. Why is this grid a thousand times smaller than ridge's?

In [ ]:
lasso_pipe = ...
lasso_grid = ...

lasso_search = ...
...

print(...)
print(...)

<details>
<summary>💡 Hint</summary>

The grid key is `'lasso__alpha'` now, because that is the step's name.

</details>

<details>
<summary>✅ Solution</summary>

```python
lasso_pipe = Pipeline([('scale', StandardScaler()), ('lasso', Lasso())])
lasso_grid = {'lasso__alpha': [0.001, 0.01, 0.1, 1]}

lasso_search = GridSearchCV(lasso_pipe, lasso_grid, cv=folds, scoring='neg_root_mean_squared_error')
lasso_search.fit(train[columns], train['vol_next'])

print(lasso_search.best_params_)
print(-lasso_search.best_score_)
```

Alpha 0.1 at 0.5797, against 0.5599 for ridge. The lasso charges $|\beta|$ and ridge charges $\beta^2$; for coefficients around 0.1 the square is a hundredth of the absolute value, so the same alpha means a very different price. A grid for one is never a grid for the other.

</details>

---

### G5 · The kept set, fold by fold  ★★★★☆  · revisits S5

Fit the lasso pipeline (alpha 0.1) on the **fitting rows of each fold** and print the list of columns it keeps each time. Does the same set come back?

In [ ]:
for fit_rows, score_rows in folds.split(train):
    ...

<details>
<summary>💡 Hint 1</summary>

`block = train.iloc[fit_rows]`, fit a fresh pipeline on it, then read `named_steps['lasso'].coef_`.

</details>

<details>
<summary>💡 Hint 2</summary>

Build the list of kept names with a loop over `zip(columns, coefs)` and an `if`, or with `[n for n, b in zip(columns, coefs) if b != 0]`.

</details>

<details>
<summary>✅ Solution</summary>

```python
for fit_rows, score_rows in folds.split(train):
    block = train.iloc[fit_rows]
    fold_lasso = Pipeline([('scale', StandardScaler()), ('lasso', Lasso(alpha=0.1))])
    fold_lasso.fit(block[columns], block['vol_next'])
    fold_coefs = fold_lasso.named_steps['lasso'].coef_
    kept = []
    for name, b in zip(columns, fold_coefs):
        if b != 0:
            kept.append(name)
    print(kept)
```

Five different lists. Only XOM_vol survives every block. Among near-copies the lasso keeps one and drops the rest, and which one depends on the rows, so a zero is not a verdict on the column.

</details>

---

### G6 · How many survive as alpha grows  ★★★★☆  · revisits S2

For alpha in `[0.01, 0.03, 0.1, 0.3, 1]`, fit the lasso pipeline and store the number of **non-zero** coefficients in a dictionary. Print it.

In [ ]:
survivors = {}

for alpha in [0.01, 0.03, 0.1, 0.3, 1]:
    ...

print(survivors)

<details>
<summary>💡 Hint</summary>

`int((candidate.named_steps['lasso'].coef_ != 0).sum())` counts the survivors as a plain number. Give each fitted pipeline its own name so `lasso` from G1 is kept.

</details>

<details>
<summary>✅ Solution</summary>

```python
survivors = {}

for alpha in [0.01, 0.03, 0.1, 0.3, 1]:
    candidate = Pipeline([('scale', StandardScaler()), ('lasso', Lasso(alpha=alpha))])
    candidate.fit(train[columns], train['vol_next'])
    survivors[alpha] = int((candidate.named_steps['lasso'].coef_ != 0).sum())

print(survivors)
```

13, 7, 5, 2 and then 0. At alpha 1 every coefficient is zero and the forecast is the training average. The lasso path from the lecture is this dictionary drawn as lines.

</details>

---

## 🧬 H · Elastic net

Both penalties at once, and a grid with two keys.

### H1 · Both penalties  ★★☆☆☆

Fit a pipeline with `ElasticNet(alpha=0.1, l1_ratio=0.5)`. Print the test RMSE and the number of non-zero coefficients.

In [ ]:
enet = Pipeline([...])
...

print('test RMSE:', ...)
print('non-zero :', ...)

<details>
<summary>💡 Hint</summary>

Name the step `'enet'`. The coefficients are `enet.named_steps['enet'].coef_`.

</details>

<details>
<summary>✅ Solution</summary>

```python
enet = Pipeline([('scale', StandardScaler()), ('enet', ElasticNet(alpha=0.1, l1_ratio=0.5))])
enet.fit(train[columns], train['vol_next'])

print('test RMSE:', rmse(test['vol_next'], enet.predict(test[columns])))
print('non-zero :', (enet.named_steps['enet'].coef_ != 0).sum())
```

0.2118 with 7 columns kept: some zeros, as with the lasso, and the rest shrunk, as with ridge.

</details>

---

### H2 · A grid with two keys  ★★★☆☆

Search `alpha` over `[0.01, 0.1, 1]` and `l1_ratio` over `[0.1, 0.5, 0.9]` at the same time. Print the best pair, the best score, and how many combinations were tried.

In [ ]:
enet = Pipeline([('scale', StandardScaler()), ('enet', ElasticNet())])
grid = {...}

search = ...
...

print(...)
print(...)
print('combinations:', ...)

<details>
<summary>💡 Hint 1</summary>

Two keys in one dictionary: `{'enet__alpha': [...], 'enet__l1_ratio': [...]}`.

</details>

<details>
<summary>💡 Hint 2</summary>

`len(search.cv_results_['params'])` is the number of combinations.

</details>

<details>
<summary>✅ Solution</summary>

```python
enet = Pipeline([('scale', StandardScaler()), ('enet', ElasticNet())])
grid = {'enet__alpha': [0.01, 0.1, 1], 'enet__l1_ratio': [0.1, 0.5, 0.9]}

search = GridSearchCV(enet, grid, cv=folds, scoring='neg_root_mean_squared_error')
search.fit(train[columns], train['vol_next'])

print(search.best_params_)
print(-search.best_score_)
print('combinations:', len(search.cv_results_['params']))
```

Alpha 0.1 with l1_ratio 0.5, at 0.5729, from 9 combinations and 45 fits. Two keys multiply; three keys would multiply again, which is why grids stay short.

</details>

---

### H3 · l1_ratio one is the lasso  ★★★★☆  · revisits S3

Fit two scaling pipelines, one with `ElasticNet(alpha=0.1, l1_ratio=1.0)` and one with `Lasso(alpha=0.1)`, and print the largest absolute difference between their coefficient arrays. Then write an assertion that it is below `1e-6`.

In [ ]:
enet1 = ...
...
lasso1 = ...
...

gap = ...
print(gap)
assert ...

<details>
<summary>💡 Hint</summary>

`np.abs(a - b).max()` on the two `coef_` arrays, then `assert gap < 1e-6`.

</details>

<details>
<summary>✅ Solution</summary>

```python
enet1 = Pipeline([('scale', StandardScaler()), ('enet', ElasticNet(alpha=0.1, l1_ratio=1.0))])
enet1.fit(train[columns], train['vol_next'])
lasso1 = Pipeline([('scale', StandardScaler()), ('lasso', Lasso(alpha=0.1))])
lasso1.fit(train[columns], train['vol_next'])

gap = np.abs(enet1.named_steps['enet'].coef_ - lasso1.named_steps['lasso'].coef_).max()
print(gap)
assert gap < 1e-6
```

Zero, or within 1e-16 of it: the same model. `l1_ratio=1` switches the ridge part off, and `l1_ratio=0` would switch the lasso part off. An assertion is a check that shouts if it is ever wrong, and it costs one line.

</details>

---

## 🔧 I · The arguments

Reading the settings, and the three you will actually change.

### I1 · What the defaults are  ★☆☆☆☆

Print `max_iter` from `Lasso().get_params()`, and the value of `'ridge__alpha'` from the `get_params()` of a scaling pipeline whose ridge has `alpha=1000`.

In [ ]:
print(...)
print(...)

<details>
<summary>💡 Hint</summary>

`get_params()` returns a dictionary; read a key from it with square brackets. The pipeline can be built inside the print.

</details>

<details>
<summary>✅ Solution</summary>

```python
print(Lasso().get_params()['max_iter'])
print(Pipeline([('scale', StandardScaler()), ('ridge', Ridge(alpha=1000))]).get_params()['ridge__alpha'])
```

1000 passes by default, and the pipeline reports its ridge's alpha under the same `step__argument` name a grid uses.

</details>

---

### I2 · Provoke the warning, then cure it  ★★★☆☆  · revisits S1

Fit `Lasso(alpha=0.001, max_iter=50)` in a scaling pipeline and watch the `ConvergenceWarning` appear. Then fit again with enough passes for it to go away, and print both test RMSEs.

In [ ]:
few = Pipeline([('scale', StandardScaler()), ('lasso', Lasso(alpha=0.001, max_iter=50))])
few.fit(train[columns], train['vol_next'])
print('50 passes  :', rmse(test['vol_next'], few.predict(test[columns])))

enough = ...
...
print('more passes:', ...)

<details>
<summary>💡 Hint</summary>

The warning names the cure. `Lasso(alpha=0.001, max_iter=5000)` is enough here.

</details>

<details>
<summary>✅ Solution</summary>

```python
few = Pipeline([('scale', StandardScaler()), ('lasso', Lasso(alpha=0.001, max_iter=50))])
few.fit(train[columns], train['vol_next'])
print('50 passes  :', rmse(test['vol_next'], few.predict(test[columns])))

enough = Pipeline([('scale', StandardScaler()), ('lasso', Lasso(alpha=0.001, max_iter=5000))])
enough.fit(train[columns], train['vol_next'])
print('more passes:', rmse(test['vol_next'], enough.predict(test[columns])))
```

The first fit prints a warning and returns a result anyway; the second is silent. The numbers are close, which is the dangerous part: a model that did not converge looks like one that did. Read your warnings.

</details>

---

### I3 · Force the signs  ★★☆☆☆

Fit a scaling pipeline with `Ridge(alpha=1000, positive=True)`. Print the test RMSE and count the negative coefficients.

In [ ]:
pos = Pipeline([...])
...

print('test RMSE:', ...)
print('negative :', ...)

<details>
<summary>💡 Hint</summary>

`positive=True` is an argument of `Ridge`. Count with `(coefs < 0).sum()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
pos = Pipeline([('scale', StandardScaler()), ('ridge', Ridge(alpha=1000, positive=True))])
pos.fit(train[columns], train['vol_next'])

print('test RMSE:', rmse(test['vol_next'], pos.predict(test[columns])))
print('negative :', (pos.named_steps['ridge'].coef_ < 0).sum())
```

0.2299 with 0 negative coefficients. Use it when you know the sign, as with portfolio weights that cannot be short; here it costs almost nothing because ridge had already made the wrong signs tiny.

</details>

---

### I4 · Why the intercept matters  ★★★☆☆

Fit the scaling-plus-ridge pipeline twice, once as usual and once with `fit_intercept=False`, and print the two test RMSEs. Explain the difference in one sentence.

In [ ]:
usual = ...
...
print('with intercept   :', ...)

noint = ...
...
print('without intercept:', ...)

<details>
<summary>💡 Hint</summary>

`Ridge(alpha=1000, fit_intercept=False)`. The columns are standardised to mean zero, but the target is not.

</details>

<details>
<summary>✅ Solution</summary>

```python
usual = Pipeline([('scale', StandardScaler()), ('ridge', Ridge(alpha=1000))])
usual.fit(train[columns], train['vol_next'])
print('with intercept   :', rmse(test['vol_next'], usual.predict(test[columns])))

noint = Pipeline([('scale', StandardScaler()), ('ridge', Ridge(alpha=1000, fit_intercept=False))])
noint.fit(train[columns], train['vol_next'])
print('without intercept:', rmse(test['vol_next'], noint.predict(test[columns])))
```

0.2181 against 0.9012. With the columns centred at zero and no intercept, the forecast for an average day is zero volatility, which is nonsense. The intercept carries the level of the target, and that is why the penalty leaves it alone.

</details>

---

## 📖 J · Reading the result

What the coefficients mean now, and what the final numbers are. Each of these fits what it needs; none depends on an earlier section.

### J1 · Say what a coefficient means  ★★☆☆☆  · revisits S1

Fit the scaling-plus-ridge pipeline (alpha 1000) and print one sentence with an f-string: the standardised coefficient on `vol_5d` to three decimals, and what a one-standard-deviation rise in `vol_5d` does to the forecast.

In [ ]:
ridge_pipe = ...
...
coefs = ...

sentence = ...
print(sentence)

<details>
<summary>💡 Hint</summary>

`coefs[columns.index('vol_5d')]` is the number. `f'...{value:.3f}...'` rounds inside the string.

</details>

<details>
<summary>✅ Solution</summary>

```python
ridge_pipe = Pipeline([('scale', StandardScaler()), ('ridge', Ridge(alpha=1000))])
ridge_pipe.fit(train[columns], train['vol_next'])
coefs = ridge_pipe.named_steps['ridge'].coef_

b = coefs[columns.index('vol_5d')]
sentence = (f'One standard deviation more volatility over the last five days '
            f'raises the forecast by {b:.3f} percentage points, other columns held fixed.')
print(sentence)
```

The coefficient is 0.079. It is a forecasting statement, not a causal one: the model was shrunk on purpose, so there is no standard error to put next to it.

</details>

---

### J2 · A small OLS on the columns the lasso kept  ★★★☆☆  · revisits S5

The lasso with alpha 0.1 keeps five columns: `vol_5d`, `vol_10d`, `ret_5d`, `AAPL_vol` and `XOM_vol`. Fit an ordinary least squares on just those five and print its test RMSE next to the lasso's.

In [ ]:
kept = ['vol_5d', 'vol_10d', 'ret_5d', 'AAPL_vol', 'XOM_vol']

small = LinearRegression()
...
lasso_pipe = ...
...

print('OLS on five columns:', ...)
print('the lasso         :', ...)

<details>
<summary>💡 Hint</summary>

Fit `small` on `train[kept]` and score on `test[kept]`; fit the lasso pipeline on all `columns` as in section G.

</details>

<details>
<summary>✅ Solution</summary>

```python
kept = ['vol_5d', 'vol_10d', 'ret_5d', 'AAPL_vol', 'XOM_vol']

small = LinearRegression()
small.fit(train[kept], train['vol_next'])
lasso_pipe = Pipeline([('scale', StandardScaler()), ('lasso', Lasso(alpha=0.1))])
lasso_pipe.fit(train[columns], train['vol_next'])

print('OLS on five columns:', rmse(test['vol_next'], small.predict(test[kept])))
print('the lasso         :', rmse(test['vol_next'], lasso_pipe.predict(test[columns])))
```

0.2044, against 0.2166 for the lasso itself: better, because once the columns are chosen the shrinkage only costs accuracy. A small unpenalised model on chosen columns is also where standard errors and t-tests live. What would be dishonest is reading them as if the five columns had been chosen in advance: the lasso picked them from these same training rows.

</details>

---

### J3 · Six forecasts, ranked  ★★★☆☆  · revisits S2

Put the test RMSE of all six forecasts from the lecture into one dictionary: the training average, persistence, one column, nineteen columns with OLS, with ridge (alpha 1000, scaled) and with lasso (alpha 0.1, scaled). Print them from best to worst.

In [ ]:
six = {}

...

for name in sorted(six, key=six.get):
    print(f'{name:26} {six[name]:.4f}')

<details>
<summary>💡 Hint 1</summary>

Each entry is one model fitted and scored, as in the earlier sections. The average and persistence need no fitting.

</details>

<details>
<summary>💡 Hint 2</summary>

`sorted(six, key=six.get)` orders the keys by their values.

</details>

<details>
<summary>✅ Solution</summary>

```python
six = {}

six['guess the average'] = rmse(test['vol_next'], np.full(len(test), train['vol_next'].mean()))
six['repeat this month'] = rmse(test['vol_next'], test['vol_20d'])

one_col = LinearRegression()
one_col.fit(train[['vol_20d']], train['vol_next'])
six['one column'] = rmse(test['vol_next'], one_col.predict(test[['vol_20d']]))

all_ols = LinearRegression()
all_ols.fit(train[columns], train['vol_next'])
six['nineteen columns, OLS'] = rmse(test['vol_next'], all_ols.predict(test[columns]))

ridge_pipe = Pipeline([('scale', StandardScaler()), ('ridge', Ridge(alpha=1000))])
ridge_pipe.fit(train[columns], train['vol_next'])
six['nineteen columns, ridge'] = rmse(test['vol_next'], ridge_pipe.predict(test[columns]))

lasso_pipe = Pipeline([('scale', StandardScaler()), ('lasso', Lasso(alpha=0.1))])
lasso_pipe.fit(train[columns], train['vol_next'])
six['nineteen columns, lasso'] = rmse(test['vol_next'], lasso_pipe.predict(test[columns]))

for name in sorted(six, key=six.get):
    print(f'{name:26} {six[name]:.4f}')
```

Best is "nineteen columns, lasso" at 0.2166 and worst is "guess the average" at 0.2906. The two penalised models are the only ones to beat the single column, and nineteen unpenalised columns sit below it.

</details>

---

### J4 · One function for any pipeline  ★★★★☆  · revisits S2

Write `evaluate(p)`: it cross-validates the pipeline `p` on the training rows with `folds`, refits it on all of them, and returns the pair `(cv_rmse, test_rmse)`. Run it on a ridge pipeline (alpha 1000) and a lasso pipeline (alpha 0.1).

In [ ]:
def evaluate(p):
    ...

ridge_pipe = ...
lasso_pipe = ...

print('ridge:', ...)
print('lasso:', ...)

<details>
<summary>💡 Hint 1</summary>

Inside: `cross_val_score(...)` for the first number, then `p.fit(...)` and `rmse(...)` on the test rows for the second.

</details>

<details>
<summary>💡 Hint 2</summary>

`return round(float(-scores.mean()), 4), round(test_rmse, 4)` returns a pair.

</details>

<details>
<summary>✅ Solution</summary>

```python
def evaluate(p):
    scores = cross_val_score(p, train[columns], train['vol_next'],
                             cv=folds, scoring='neg_root_mean_squared_error')
    p.fit(train[columns], train['vol_next'])
    test_rmse = rmse(test['vol_next'], p.predict(test[columns]))
    return round(float(-scores.mean()), 4), round(test_rmse, 4)

ridge_pipe = Pipeline([('scale', StandardScaler()), ('ridge', Ridge(alpha=1000))])
lasso_pipe = Pipeline([('scale', StandardScaler()), ('lasso', Lasso(alpha=0.1))])

print('ridge:', evaluate(ridge_pipe))
print('lasso:', evaluate(lasso_pipe))
```

Ridge (0.5599, 0.2181) and lasso (0.5797, 0.2166). One function, any pipeline: because every scikit-learn model fits and predicts the same way, the function never needs to know what is inside.

</details>

---

### J5 · The worst day  ★★★★☆  · revisits S5

Fit the ridge pipeline (alpha 1000) and find the test day on which its forecast was furthest from what happened. Print the date, what happened, and the forecast.

In [ ]:
ridge_pipe = ...
...
forecast = ...

worst = ...
print(..., ..., ...)

<details>
<summary>💡 Hint 1</summary>

`errors = np.abs(test['vol_next'].values - forecast)` is the size of every miss; `errors.argmax()` is the position of the largest.

</details>

<details>
<summary>💡 Hint 2</summary>

`test.index[worst].date()`, `test['vol_next'].iloc[worst]` and `forecast[worst]` read the three things off at that position.

</details>

<details>
<summary>✅ Solution</summary>

```python
ridge_pipe = Pipeline([('scale', StandardScaler()), ('ridge', Ridge(alpha=1000))])
ridge_pipe.fit(train[columns], train['vol_next'])
forecast = ridge_pipe.predict(test[columns])

errors = np.abs(test['vol_next'].values - forecast)
worst = errors.argmax()
print(test.index[worst].date(), round(test['vol_next'].iloc[worst], 3), round(forecast[worst], 3))
```

2024-07-16: the next twenty days turned out at 1.38 and the forecast said 0.72. Look up what happened in the market in the weeks after that date; a volatility model made from past returns cannot see an event coming, and the biggest misses are always the days before one.

</details>

---

## 🔁 K · Across the desk

The whole workflow, once per instrument. This section uses `build_table` from A7; if you skipped A7, copy its solution into the first cell. Each exercise builds the tables it needs.

### K1 · Nvidia, with everything  ★★★★☆  · revisits S2

Use your `build_table` from A7 on `'NVDA'`, the most volatile stock in the data. Split at the end of 2022, then print three test RMSEs: one column (`vol_20d`), nineteen columns with OLS, and nineteen columns with a ridge whose alpha `GridSearchCV` chooses from `[1, 10, 100, 1000, 10000]`.

In [ ]:
nvda = build_table('NVDA')
tr = ...
te = ...
cols = ...

...

print('one column        :', ...)
print('all columns, OLS  :', ...)
print('all columns, ridge:', ...)

<details>
<summary>💡 Hint 1</summary>

`cols = list(nvda.columns[:-1])`, then the three fits from earlier sections with `tr` and `te` in place of `train` and `test`.

</details>

<details>
<summary>💡 Hint 2</summary>

The search is F3 with the pipeline and grid unchanged.

</details>

<details>
<summary>✅ Solution</summary>

```python
nvda = build_table('NVDA')
tr = nvda.loc[:'2022-12-31']
te = nvda.loc['2023-01-01':]
cols = list(nvda.columns[:-1])

one_n = LinearRegression()
one_n.fit(tr[['vol_20d']], tr['vol_next'])
ols_n = LinearRegression()
ols_n.fit(tr[cols], tr['vol_next'])
search_n = GridSearchCV(Pipeline([('scale', StandardScaler()), ('ridge', Ridge())]),
                        {'ridge__alpha': [1, 10, 100, 1000, 10000]},
                        cv=folds, scoring='neg_root_mean_squared_error')
search_n.fit(tr[cols], tr['vol_next'])

print('one column        :', rmse(te['vol_next'], one_n.predict(te[['vol_20d']])))
print('all columns, OLS  :', rmse(te['vol_next'], ols_n.predict(te[cols])))
print('all columns, ridge:', rmse(te['vol_next'], search_n.predict(te[cols])))
```

1.137, 1.142 and 1.110, with alpha 10,000 chosen. The numbers are five times the index's because Nvidia moves five times as much, and the order is the same: the penalised nineteen beat the single column, and the unpenalised nineteen do not.

</details>

---

### K2 · Which alpha does each instrument pick  ★★★★★  · revisits S2

For every ticker in `rets`, build its table with `build_table`, run the same grid search on the training rows, and store the winning alpha in a dictionary `best_alpha`. Print it, and count how many instruments pick 1000.

In [ ]:
best_alpha = {}

for ticker in rets.columns:
    ...

print(best_alpha)
print('pick 1000:', ...)

<details>
<summary>💡 Hint 1</summary>

The loop body is K1's search with `ticker` in place of `'NVDA'`, ending in `best_alpha[ticker] = search.best_params_['ridge__alpha']`.

</details>

<details>
<summary>💡 Hint 2</summary>

`sum(1 for t in best_alpha if best_alpha[t] == 1000)` counts. Eleven searches take a little while.

</details>

<details>
<summary>✅ Solution</summary>

```python
best_alpha = {}

for ticker in rets.columns:
    frame = build_table(ticker)
    tr = frame.loc[:'2022-12-31']
    cols = list(frame.columns[:-1])
    search_t = GridSearchCV(Pipeline([('scale', StandardScaler()), ('ridge', Ridge())]),
                            {'ridge__alpha': [1, 10, 100, 1000, 10000]},
                            cv=folds, scoring='neg_root_mean_squared_error')
    search_t.fit(tr[cols], tr['vol_next'])
    best_alpha[ticker] = search_t.best_params_['ridge__alpha']

print(best_alpha)
print('pick 1000:', sum(1 for t in best_alpha if best_alpha[t] == 1000))
```

9 of 11 pick 1000; KO picks 10,000, NVDA picks 10,000. One grid, one factor of ten apart, and nearly the same answer on every instrument: with standardised columns the right strength of penalty is a property of the problem, not of the stock. The test rows were never touched.

</details>

---

### K3 · How many columns the lasso keeps, per instrument  ★★★☆☆  · revisits S3

For every ticker, fit the scaling-plus-`Lasso(alpha=0.1)` pipeline on its training rows and count the non-zero coefficients. Draw the counts as a bar chart, sorted from most to fewest.

In [ ]:
survivors = {}
for ticker in rets.columns:
    ...

ranked = ...

fig, ax = plt.subplots(figsize=(8, 3.2))
...
plt.show()

<details>
<summary>💡 Hint 1</summary>

`survivors[ticker] = int((lasso_t.named_steps['lasso'].coef_ != 0).sum())`.

</details>

<details>
<summary>💡 Hint 2</summary>

`ranked = pd.Series(survivors).sort_values(ascending=False)`, then `ax.bar(ranked.index, ranked.values)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
survivors = {}
for ticker in rets.columns:
    frame = build_table(ticker)
    tr = frame.loc[:'2022-12-31']
    cols = list(frame.columns[:-1])
    lasso_t = Pipeline([('scale', StandardScaler()), ('lasso', Lasso(alpha=0.1))])
    lasso_t.fit(tr[cols], tr['vol_next'])
    survivors[ticker] = int((lasso_t.named_steps['lasso'].coef_ != 0).sum())

ranked = pd.Series(survivors).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.bar(ranked.index, ranked.values)
ax.set_ylabel('columns kept of 19')
ax.set_title('Lasso, alpha 0.1: how many columns survive, per instrument', loc='left')
plt.show()
```

From 8 on AAPL down to 2 on JNJ. The same alpha keeps a different number of columns on each instrument, because the lasso's alpha is a price in the units of the target, and the targets differ in size. A lasso grid, unlike ridge's, has to be searched per problem.

</details>

---

## 🏁 Done

You built the lecture's table from the price file, watched nineteen unpenalised columns forecast worse than one, and fixed it: a penalty, the scaling it needs, a pipeline so the folds refit both, and a grid search to set the penalty's strength. Then you did it eleven times.

The case takes the same tools back to the risk report, where the data is in decimals and the results are not the same on every stock.